<a href="https://colab.research.google.com/github/tukamilano/Semi-Thue/blob/main/bidirectional_bfs.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [36]:
import random

alphabet_num = 10
length = 20 # 長さは固定

alphabet = tuple(chr(i + 65) for i in range(alphabet_num))

# ルールは幾何分布を使ってランダムに生成
rule_num = 100
rule_length_expect_inv = 0.5 #幾何分布のパラメータpにあたる

rule = []
while len(rule) < rule_num:
    i = 1
    while (random.random() > rule_length_expect_inv) and (i < length):
        i += 1
    before = ''.join(random.choices(alphabet, k=i))
    after = ''.join(random.choices(alphabet, k=i))
    if (before != after) and ((before, after) not in rule): # これは必要か?
        rule.append((before, after))

print(rule)

[('CEIF', 'HIFG'), ('IC', 'FH'), ('DD', 'JD'), ('C', 'E'), ('HB', 'EE'), ('C', 'H'), ('EH', 'JE'), ('JCFD', 'DHGD'), ('FII', 'BBF'), ('J', 'A'), ('J', 'G'), ('G', 'E'), ('HJ', 'GH'), ('D', 'A'), ('D', 'G'), ('AH', 'DB'), ('FA', 'AI'), ('JE', 'FJ'), ('H', 'C'), ('F', 'G'), ('D', 'J'), ('F', 'C'), ('I', 'J'), ('D', 'I'), ('CB', 'AA'), ('GD', 'GE'), ('C', 'I'), ('JE', 'GH'), ('B', 'J'), ('A', 'H'), ('FAGEA', 'GGCHB'), ('HADAJ', 'AIAHJ'), ('B', 'I'), ('I', 'C'), ('GJ', 'AA'), ('A', 'F'), ('ID', 'CE'), ('BHCDI', 'AGGAJ'), ('D', 'H'), ('H', 'J'), ('A', 'C'), ('J', 'D'), ('AF', 'CA'), ('CB', 'HA'), ('HAJJAJ', 'IIJFIB'), ('GDG', 'CCF'), ('E', 'A'), ('BG', 'EA'), ('I', 'F'), ('F', 'I'), ('A', 'J'), ('B', 'E'), ('HFF', 'EEC'), ('J', 'I'), ('DJJ', 'IJG'), ('IH', 'GC'), ('FEID', 'DAEH'), ('B', 'C'), ('B', 'A'), ('C', 'F'), ('BJE', 'AFH'), ('H', 'F'), ('D', 'C'), ('AABAG', 'IGCCJ'), ('HAC', 'CCH'), ('H', 'B'), ('BGG', 'HCI'), ('DA', 'CA'), ('C', 'J'), ('B', 'H'), ('CDAE', 'BEEG'), ('G', 'J'), ('EGG

In [37]:
while True:
  before = ''.join(random.choices(alphabet, k=length))
  after = ''.join(random.choices(alphabet, k=length))
  if before != after:
    break

print(before)
print(after)
# ruleを使って到達可能か?

ABEDHGDAFGIDACHADGDJ
EHDEABIFIECHHEIECJEE


In [38]:
from collections import deque

def bidirectional_bfs_path(before, after, rules):
    forward_q = deque([before])
    backward_q = deque([after])
    forward_searched = set([before])
    backward_searched = set([after])

    # 各状態の親（遷移元）を記録
    forward_parents = {before: None}
    backward_parents = {after: None}

    # どちらの方向で交差したかを記録
    meeting_point = None

    while forward_q and backward_q:
        # 小さい方のキューを進める
        if len(forward_q) <= len(backward_q):
            current_q = forward_q
            current_searched = forward_searched
            opposite_searched = backward_searched
            current_parents = forward_parents
            direction = "forward"
        else:
            current_q = backward_q
            current_searched = backward_searched
            opposite_searched = forward_searched
            current_parents = backward_parents
            direction = "backward"

        state = current_q.pop()
        for rule_before, rule_after in rules:
            # 適用方向によってルールを切り替える
            if direction == "forward":
                src, dst = rule_before, rule_after
            else:
                src, dst = rule_after, rule_before

            i = len(src)
            for j in range(len(state) - i + 1):
                if state[j:j+i] == src:
                    new_state = state[:j] + dst + state[j+i:]
                    if new_state in current_searched:
                        continue
                    current_searched.add(new_state)
                    current_q.appendleft(new_state)
                    current_parents[new_state] = state

                    if new_state in opposite_searched:
                        meeting_point = new_state
                        return reconstruct_path(forward_parents, backward_parents, meeting_point)

    print("到達不可")
    return None

def reconstruct_path(forward_parents, backward_parents, meeting_point):
    # 前向きパス（start -> meeting_point）
    path_f = []
    node = meeting_point
    while node is not None:
        path_f.append(node)
        node = forward_parents.get(node)
    path_f.reverse()

    # 後ろ向きパス（meeting_point -> goal）
    path_b = []
    node = backward_parents.get(meeting_point)
    while node is not None:
        path_b.append(node)
        node = backward_parents.get(node)

    # 合体
    return path_f + path_b



In [39]:
path = bidirectional_bfs_path(before, after, rule)
if path:
    print("最短変換系列:")
    for step in path:
        print(step)

KeyboardInterrupt: 